## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Small Datasets, Part 1: From Raw Force Signals to Gait Features

In biomedical applications, datasets often have only a few subjects. With a small dataset, it is difficult to develop a general model for classification (or regression), because the data cannot show all the variation between people.

But small datasets are still useful. We can use them as a **proof of principle**: a first result that shows that it is worth collecting more data.

There is no exact definition of a *SMALL* dataset. The scikit-learn guide for selecting a model starts with one question: "Do you have more than 50 samples?" If the answer is no, the guide says: **get more data** 😨.

When one subject gives many samples, count the **subjects**, not the samples. Samples from the same person are not independent (notebook 04.03).

## The data

In these notebooks, we use a small dataset from this study:

>Silvi Frenkel-Toledo, Nir Giladi, Chava Peretz, Talia Herman, Leor Gruendlinger, Jeffrey M. Hausdorff (2005). Treadmill walking as an external pacemaker to improve gait rhythm and stability in Parkinson's disease. *Movement Disorders*, 20(9), 1109-1114.

Persons with Parkinson's disease (PD) and healthy controls walked on level ground, at their self-selected speed, for about 2 minutes. They wore instrumented insoles. Each insole has 8 force sensors. The data has the force of each sensor (16 channels) and the total force under each foot (2 more channels), sampled at 100 Hz.

Other studies used the same sensors with persons with PD. The data is available at PhysioNet: https://physionet.org/content/gaitpdb/1.0.0/

## What we will do

In this notebook, we use the force signals to compute gait features. In Part 2, we use these features to train a classifier that separates controls from persons with PD.

We will:
- look at the data
- detect when each foot is on the ground
- find the heel strikes and the toe-offs
- compute the stride, stance, swing and step times, and the cadence
- put all the steps in one function, and check it against the feature table that we use in Part 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# load one recording: a control subject (Co = control, Pt = patient)
columns = [
    "time",
    "L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8",
    "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
    "L_total", "R_total"
]

sample_file = "data/SiCo01_01.txt"
df = pd.read_csv(sample_file, sep=r"\s+", header=None, names=columns)
t = df["time"].values

print("Rows:", len(df))
print(f"Duration: {t[-1]:.1f} s, sampling interval: {t[1] - t[0]:.2f} s ({1 / (t[1] - t[0]):.0f} Hz)")
df.head()

In [ ]:
# plot the total force under each foot, first 5 seconds
first = t <= 5
plt.figure(figsize=(12, 4))
plt.plot(t[first], df["L_total"][first], label="Left total force")
plt.plot(t[first], df["R_total"][first], label="Right total force")
plt.xlabel("Time (s)")
plt.ylabel("Force (N)")
plt.title("Total vertical force under each foot")
plt.legend()
plt.tight_layout()
plt.show()

### What can we compute from this data?

The force is high when the foot is on the ground and close to zero when the foot is in the air. So the force tells us when the foot touches the ground (**heel strike**) and when it leaves the ground (**toe-off**). From these two events we can compute:

- stride time
- stance time
- swing time
- step time
- cadence

<div style="text-align:center;">
    <img src="images/gaitcyle.jpg"
    style="max-width:80%; height:auto;">
</div>

The insoles do **not** measure the step length: they only measure force. To get the step length, we also need the walking speed. The study measured the speed of each subject separately. Then step length = speed × step time.

### Select a threshold

We detect the contact with a threshold: the foot is on the ground when the force is above the threshold. We try a few values.

In [ ]:
first = t <= 10
plt.figure(figsize=(12, 4))
plt.plot(t[first], df["R_total"][first], color='tab:orange', alpha=0.7, label="Right total force")
for thr, color in zip([5, 10, 20], ["blue", "green", "red"]):
    plt.axhline(thr, linestyle="--", color=color, label=f"{thr} N")
plt.xlabel("Time (s)")
plt.ylabel("Force (N)")
plt.title("Possible contact thresholds (zoom on low forces)")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

- **5 N and 10 N:** too close to 0. Noise in the swing phase can go above these values.
- **20 N:** above the noise, and much lower than the force during stance. We use 20 N and check the result.

In [ ]:
threshold = 20.0

# 1 when the foot is on the ground, 0 when it is in the air
left_contact = (df["L_total"].values > threshold).astype(int)
right_contact = (df["R_total"].values > threshold).astype(int)

first = t <= 5
plt.figure(figsize=(12, 4))
plt.plot(t[first], left_contact[first], label="Left contact")
plt.plot(t[first], right_contact[first], '--', label="Right contact")
plt.xlabel("Time (s)")
plt.ylabel("Ground contact (1 = yes, 0 = no)")
plt.title("Foot contact detection")
plt.legend()
plt.tight_layout()
plt.show()

When the foot touches the ground, the contact signal changes from 0 to 1. When the foot leaves the ground, it changes from 1 to 0. The difference between consecutive samples (`np.diff`) finds these changes:

- `+1` (0 → 1) = heel strike
- `-1` (1 → 0) = toe-off

`np.where` then gives the positions (indices) of these events.

In [ ]:
first = t[:-1] <= 5
plt.figure(figsize=(12, 4))
plt.plot(t[:-1][first], np.diff(left_contact)[first], label="Left foot events")
plt.plot(t[:-1][first], np.diff(right_contact)[first], '--', label="Right foot events")
plt.xlabel("Time (s)")
plt.ylabel("Event (+1 = heel strike, -1 = toe-off)")
plt.title("Gait events")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# +1 because np.diff(x)[i] compares sample i and sample i+1: the change happens at sample i+1
heel_strikes_left = np.where(np.diff(left_contact) == 1)[0] + 1
toe_offs_left = np.where(np.diff(left_contact) == -1)[0] + 1
heel_strikes_right = np.where(np.diff(right_contact) == 1)[0] + 1
toe_offs_right = np.where(np.diff(right_contact) == -1)[0] + 1

print("Left : heel strikes =", len(heel_strikes_left), "| toe-offs =", len(toe_offs_left))
print("Right: heel strikes =", len(heel_strikes_right), "| toe-offs =", len(toe_offs_right))
print(f"\nLeft foot:  first heel strike at {t[heel_strikes_left[0]]:.2f} s, first toe-off at {t[toe_offs_left[0]]:.2f} s")
print(f"Right foot: first heel strike at {t[heel_strikes_right[0]]:.2f} s, first toe-off at {t[toe_offs_right[0]]:.2f} s")

### Be careful with the order of the events

The recording starts with **both feet on the ground**, so the **first event of each foot is a toe-off**, not a heel strike. If we pair `heel_strikes[i]` with `toe_offs[i]` without a check, we pair events from different strides, and the times are wrong (for example, a negative swing time).

To prevent this, we make every stride start with a heel strike:

1. Remove the toe-offs that come before the first heel strike.
2. Keep only complete strides: heel strike → toe-off → next heel strike.

Then the events are in this order: `HS[0] < TO[0] < HS[1] < TO[1] < HS[2] ...`

In [ ]:
def order_events(heel_strikes, toe_offs):
    # make every stride start with a heel strike: HS[i] < TO[i] < HS[i+1]
    toe_offs = toe_offs[toe_offs > heel_strikes[0]]        # 1. remove toe-offs before the first heel strike
    n_strides = min(len(heel_strikes) - 1, len(toe_offs))  # 2. keep only complete strides
    heel_strikes = heel_strikes[:n_strides + 1]            #    n strides need n+1 heel strikes
    toe_offs = toe_offs[:n_strides]
    # check the order
    assert np.all(heel_strikes[:-1] < toe_offs) and np.all(toe_offs < heel_strikes[1:])
    return heel_strikes, toe_offs

heel_strikes_left, toe_offs_left = order_events(heel_strikes_left, toe_offs_left)
heel_strikes_right, toe_offs_right = order_events(heel_strikes_right, toe_offs_right)
print("Left strides :", len(toe_offs_left))
print("Right strides:", len(toe_offs_right))

In [ ]:
# check the events on the force signal
plt.figure(figsize=(12, 5))
plt.plot(t, df["L_total"], label="Left total force")
plt.plot(t, df["R_total"], label="Right total force")
plt.scatter(t[heel_strikes_left], df["L_total"].values[heel_strikes_left], color="red", label="Left heel strike", zorder=5)
plt.scatter(t[toe_offs_left], df["L_total"].values[toe_offs_left], color="orange", label="Left toe-off", zorder=5)
plt.scatter(t[heel_strikes_right], df["R_total"].values[heel_strikes_right], color="blue", label="Right heel strike", zorder=5)
plt.scatter(t[toe_offs_right], df["R_total"].values[toe_offs_right], color="cyan", label="Right toe-off", zorder=5)
plt.xlabel("Time (s)")
plt.ylabel("Force (N)")
plt.title("Total vertical force with heel strikes and toe-offs")
plt.xlim(0, 10)
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## Compute the gait measures

For one foot, with the events in the order `HS[i] < TO[i] < HS[i+1]`:

- **Stride time** = time from one heel strike to the next heel strike of the same foot: `HS[i+1] - HS[i]`
- **Stance time** = time from heel strike to toe-off of the same foot (foot on the ground): `TO[i] - HS[i]`
- **Swing time** = time from toe-off to the next heel strike of the same foot (foot in the air): `HS[i+1] - TO[i]`

So stance time + swing time = stride time.

In [ ]:
def stride_stance_swing(heel_strikes, toe_offs):
    stride = t[heel_strikes[1:]] - t[heel_strikes[:-1]]
    stance = t[toe_offs] - t[heel_strikes[:-1]]
    swing = t[heel_strikes[1:]] - t[toe_offs]
    return stride, stance, swing

stride_left, stance_left, swing_left = stride_stance_swing(heel_strikes_left, toe_offs_left)
stride_right, stance_right, swing_right = stride_stance_swing(heel_strikes_right, toe_offs_right)

print(f"Left : stride {stride_left.mean():.3f} s, stance {stance_left.mean():.3f} s, swing {swing_left.mean():.3f} s")
print(f"Right: stride {stride_right.mean():.3f} s, stance {stance_right.mean():.3f} s, swing {swing_right.mean():.3f} s")

# sanity checks: always check that your numbers make physical sense
print("\nAll times positive:", all(np.all(x > 0) for x in [stance_left, swing_left, stance_right, swing_right]))
print("stance + swing = stride:", np.allclose(stance_left + swing_left, stride_left))
print(f"Stance as % of stride: {100 * stance_left.mean() / stride_left.mean():.0f}%  (about 60 to 65% is typical for walking)")

### Step time and cadence

- **Step time** = time from the heel strike of one foot to the next heel strike of the **other** foot. Here we use the steps from a left heel strike to the next right heel strike.
- **Cadence** = steps per minute = 60 / step time.

The two feet can have a different number of events. So we do **not** pair the events by their index. For each left heel strike, we find the **next** right heel strike. `np.searchsorted(a, v)` gives the position in the sorted array `a` where the value `v` would go, so `a[position]` is the first value of `a` after `v`.

In [ ]:
t_hs_left = t[heel_strikes_left]
t_hs_right = t[heel_strikes_right]

next_right = np.searchsorted(t_hs_right, t_hs_left)   # position of the next right heel strike
valid = next_right < len(t_hs_right)                  # the last left heel strike can have no next right heel strike
step_times = t_hs_right[next_right[valid]] - t_hs_left[valid]

cadence = 60 / step_times.mean()
print(f"Mean step time: {step_times.mean():.3f} s")
print(f"Cadence: {cadence:.1f} steps/min")

### Variability

Gait in PD is often less regular. A common measure of regularity is the **coefficient of variation (CV)**:

$$CV = \frac{\text{standard deviation}}{\text{mean}}$$

A CV of 0.05 means that the times change by about 5% from one stride to the next.

In [ ]:
def cv(x):
    return np.std(x) / np.mean(x)

print(f"CV of the stride time (left): {cv(stride_left):.3f}")
print(f"CV of the step time:          {cv(step_times):.3f}")

## Put all the steps in one function

We put all the steps in one function. Then we can compute the same features for every recording in the database with a loop. The function needs the walking speed of the subject to compute the step length.

In [ ]:
def compute_gait_features(file, speed, threshold=20.0):
    data = pd.read_csv(file, sep=r"\s+", header=None, names=columns)
    time = data["time"].values
    features = {"avg_speed": speed}
    hs_times = {}

    for side, name in [("L", "left"), ("R", "right")]:
        contact = (data[f"{side}_total"].values > threshold).astype(int)
        hs = np.where(np.diff(contact) == 1)[0] + 1
        to = np.where(np.diff(contact) == -1)[0] + 1
        hs, to = order_events(hs, to)
        stride = time[hs[1:]] - time[hs[:-1]]
        stance = time[to] - time[hs[:-1]]
        swing = time[hs[1:]] - time[to]
        features[f"avg_stride_time_{name}"] = stride.mean()
        features[f"avg_stance_time_{name}"] = stance.mean()
        features[f"avg_swing_time_{name}"] = swing.mean()
        features[f"cv_stride_time_{name}"] = cv(stride)
        features[f"cv_stance_time_{name}"] = cv(stance)
        features[f"cv_swing_time_{name}"] = cv(swing)
        hs_times[side] = time[hs]

    nxt = np.searchsorted(hs_times["R"], hs_times["L"])
    ok = nxt < len(hs_times["R"])
    steps = hs_times["R"][nxt[ok]] - hs_times["L"][ok]
    features["avg_step_time"] = steps.mean()
    features["cv_step_time"] = cv(steps)
    features["avg_step_length"] = speed * steps.mean()   # step length = speed x step time
    features["cadence"] = 60 / steps.mean()
    return features

### Check the function against the feature table

Part 2 uses the file `gait_measures_PD_controls.csv`. It has one row per subject, with the features of each subject's recording. We compute the features of one control (`SiCo01`) and one person with PD (`SiPt02`) with our function, and compare them with the table.

In [ ]:
table = pd.read_csv("data/gait_measures_PD_controls.csv").set_index("ID")

comparison = {}
for subject in ["SiCo01", "SiPt02"]:
    speed = table.loc[subject, "avg_speed"]      # the walking speed measured in the study
    ours = compute_gait_features(f"data/{subject}_01.txt", speed)
    comparison[(subject, "our function")] = ours
    comparison[(subject, "table")] = table.loc[subject, list(ours.keys())]

pd.DataFrame(comparison).round(3)

The values from our function are the same as the values in the table. The table was made with this method for every subject in the study.

Compare the two subjects. The person with PD (`SiPt02`) walks slower, with shorter steps, and the stride times are much more variable (larger CV), mainly on the right side.

In Part 2, we use these features for all the subjects to train a classifier.

### Try it
1. Change the threshold to 10 N and to 40 N. Do the features change much?
2. Compute the **double support time**: the time when both feet are on the ground. (Hint: `left_contact & right_contact` is 1 when both feet are on the ground.)